In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()


DB_USER = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
DB_HOST = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_NAME = os.getenv("DB_NAME", "testdb")

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, engine)

counts = q("""
SELECT 'Project' t, COUNT(*) n FROM Project
UNION ALL SELECT 'Sprint', COUNT(*) FROM Sprint
UNION ALL SELECT 'Issue', COUNT(*) FROM Issue
UNION ALL SELECT 'Change_Log', COUNT(*) FROM Change_Log
UNION ALL SELECT 'Comment', COUNT(*) FROM Comment
UNION ALL SELECT 'Issue_Link', COUNT(*) FROM Issue_Link;
""")
counts

counts.to_csv("../reports/first_look/table_counts.csv", index=False)


proj = q("""
SELECT
  p.Project_Key,
  p.Name,
  COUNT(*) AS n_issues,
  SUM(i.Resolution_Date IS NOT NULL) AS n_resolved,
  MIN(i.Creation_Date) AS min_created,
  MAX(i.Creation_Date) AS max_created,
  MIN(i.Resolution_Date) AS min_resolved,
  MAX(i.Resolution_Date) AS max_resolved
FROM Issue i
JOIN Project p ON p.ID = i.Project_ID
GROUP BY p.Project_Key, p.Name
ORDER BY n_issues DESC;
""")
proj.head(20)
proj.to_csv("../reports/first_look/project_coverage.csv", index=False)


miss = q("""
SELECT
  SUM(Creation_Date IS NULL) / COUNT(*) AS pct_creation_null,
  SUM(Resolution_Date IS NULL) / COUNT(*) AS pct_resolution_null,
  SUM(Estimation_Date IS NULL) / COUNT(*) AS pct_estimation_null,
  SUM(In_Progress_Minutes IS NULL) / COUNT(*) AS pct_inprogmin_null,
  SUM(Resolution_Time_Minutes IS NULL) / COUNT(*) AS pct_resolutiontime_null,
  SUM(Story_Point IS NULL) / COUNT(*) AS pct_storypoints_null,
  SUM(Sprint_ID IS NULL) / COUNT(*) AS pct_sprint_null
FROM Issue;
""")
miss
miss.to_csv("../reports/first_look/missingness_overall.csv", index=False)



fields = q("""
SELECT LOWER(Field) AS field, COUNT(*) n
FROM Change_Log
GROUP BY LOWER(Field)
ORDER BY n DESC;
""")
fields.head(30)
fields.to_csv("../reports/first_look/changelog_field_counts.csv", index=False)

status_vals = q("""
SELECT To_String, COUNT(*) n
FROM Change_Log
WHERE LOWER(Field) = 'status'
GROUP BY To_String
ORDER BY n DESC
LIMIT 100;
""")
status_vals
status_vals.to_csv("../reports/first_look/status_to_values_top100.csv", index=False)


def export_table(table: str):
    df = q(f"SELECT * FROM {table};")
    out = Path("data/interim") / f"{table}.parquet"
    df.to_parquet(out, index=False)
    print(f"{table}: {len(df):,} rows -> {out}")

Path("data/interim").mkdir(parents=True, exist_ok=True)
for t in ["Project", "Sprint", "Issue", "Change_Log"]:
    export_table(t)



Project: 39 rows -> data/interim/Project.parquet
Sprint: 4,594 rows -> data/interim/Sprint.parquet
Issue: 458,232 rows -> data/interim/Issue.parquet
Change_Log: 9,253,419 rows -> data/interim/Change_Log.parquet
